# 📡 Módulo 5: Despliegue de Modelos de Machine Learning
## 20. Monitorización, Mantenimiento y Re-entrenamiento

### Curso: **Machine Learning con Python** (IFCD093PO)
**Duración estimada:** 4 horas

---

## 🎯 Objetivos del Notebook

Desplegar un modelo no es el final del camino, es el principio de su vida útil. Un modelo en producción está expuesto al mundo real, un entorno dinámico que cambia constantemente. Si no lo vigilamos, su rendimiento se degradará inevitablemente. Este proceso de vigilancia y cuidado se llama **monitorización y mantenimiento**.

En este notebook, abordaremos los conceptos y técnicas para asegurar que nuestros modelos sigan siendo precisos y relevantes a lo largo del tiempo.

**Objetivos principales:**
1.  Entender por qué los modelos se degradan en producción.
2.  Diferenciar entre **Concept Drift**, **Data Drift** y **Model Drift**.
3.  Aprender qué métricas monitorizar: métricas de rendimiento del software y métricas de rendimiento del modelo.
4.  Conocer estrategias para detectar el drift.
5.  Diseñar pipelines de **re-entrenamiento** automático.

---

## 📚 Contenidos del Notebook

1. [La Realidad de la Producción: ¿Por qué se Degradan los Modelos?](#1-realidad)
2. [Tipos de "Drift" (Deriva o Desviación)](#2-drift)
   - [Concept Drift](#2.1-concept)
   - [Data Drift](#2.2-data)
   - [Upstream Data Changes](#2.3-upstream)
3. [¿Qué Debemos Monitorizar?](#3-monitorizar)
   - [Métricas de Rendimiento del Sistema/Software](#3.1-software-metrics)
   - [Métricas de Rendimiento del Modelo](#3.2-model-metrics)
4. [Estrategias para la Detección de Drift](#4-deteccion)
   - [Monitorización de la Distribución de los Datos](#4.1-dist-monitor)
   - [Uso de Tests Estadísticos (Ej. Kolmogorov-Smirnov)](#4.2-ks-test)
   - [Monitorización del Rendimiento del Modelo](#4.3-perf-monitor)
5. [Estrategias de Re-entrenamiento](#5-retraining)
   - [¿Cuándo Re-entrenar?](#5.1-cuando)
   - [Enfoques de Re-entrenamiento](#5.2-enfoques)
   - [Diseño de un Pipeline de Re-entrenamiento](#5.3-pipeline)
6. [Herramientas de Monitorización](#6-herramientas)
7. [Resumen y Cierre del Ciclo de MLOps](#7-resumen)

---

## 1. La Realidad de la Producción: ¿Por qué se Degradan los Modelos? <a id='1-realidad'></a>

Un modelo de Machine Learning aprende patrones a partir de un conjunto de datos de entrenamiento. La suposición fundamental es que los datos que verá en el futuro (en producción) seguirán la misma distribución y patrones que los datos con los que fue entrenado.

**Esta suposición casi nunca es cierta a largo plazo.**

El mundo cambia:
- El **comportamiento de los clientes** evoluciona (ej. una pandemia cambia los patrones de compra).
- Los **competidores** lanzan nuevos productos.
- Surgen **nuevos tipos de fraude**.
- La **economía** fluctúa, afectando los precios de las casas.

Cuando la realidad cambia, los patrones que el modelo aprendió dejan de ser válidos. A este fenómeno general se le llama **degradación del modelo** o **model decay**.

---

## 2. Tipos de "Drift" (Deriva o Desviación) <a id='2-drift'></a>

La degradación del modelo se puede desglosar en varios tipos de "drift".

### 2.1. Concept Drift <a id='2.1-concept'></a>

- **¿Qué es?**: La relación entre las características de entrada y la variable objetivo cambia. Los patrones que el modelo aprendió ya no son correctos.
- **Ejemplo**: Un modelo de scoring de crédito aprendió que "tener una hipoteca" era una señal de solvencia. Tras una crisis inmobiliaria, tener una hipoteca podría empezar a asociarse con un mayor riesgo. La característica es la misma, pero su significado (el concepto) ha cambiado.
- **Detección**: Es el más difícil de detectar directamente. Normalmente se infiere a través de una caída en las métricas de rendimiento del modelo (ej. el accuracy baja).

### 2.2. Data Drift <a id='2.2-data'></a>

- **¿Qué es?**: La distribución estadística de los datos de entrada cambia, aunque la relación fundamental con el objetivo no lo haga.
- **Ejemplo**: Un modelo de recomendación de ropa fue entrenado con datos de clientes cuya edad media era de 30 años. Con el tiempo, la plataforma se vuelve popular entre adolescentes, y la edad media de los nuevos usuarios baja a 18. El modelo ahora ve datos (edades) que no eran comunes en su entrenamiento, y puede que no funcione bien para este nuevo grupo demográfico.
- **Detección**: Se detecta comparando las distribuciones de las características de entrada entre los datos de entrenamiento y los datos de producción (ej. histogramas, tests estadísticos).

### 2.3. Upstream Data Changes <a id='2.3-upstream'></a>

- **¿Qué es?**: Cambios en el pipeline de datos que alimenta al modelo. No es un cambio en el mundo real, sino un problema técnico.
- **Ejemplo**: Un equipo de ingeniería cambia la unidad de una característica de metros a centímetros sin avisar. El modelo de repente recibe valores 100 veces más grandes de lo esperado, y sus predicciones serán incorrectas.
- **Detección**: Se detecta con validación de datos (data validation) y monitorización de la calidad de los datos (ej. rangos esperados, tipos de datos, valores nulos).

![Drift Types](imagenes/Drift_Types.png)

---

## 3. ¿Qué Debemos Monitorizar? <a id='3-monitorizar'></a>

La monitorización se divide en dos grandes áreas:

### 3.1. Métricas de Rendimiento del Sistema/Software <a id='3.1-software-metrics'></a>

Se centran en la salud de nuestra API como aplicación de software.

- **Latencia**: ¿Cuánto tarda la API en responder? Un aumento en la latencia puede indicar un problema.
- **Tasa de Peticiones (Throughput)**: ¿Cuántas predicciones por segundo estamos sirviendo?
- **Tasa de Errores**: ¿Qué porcentaje de peticiones fallan (errores 5xx)?
- **Uso de Recursos**: Consumo de CPU, memoria y disco del contenedor o servidor.

**Herramientas**: Prometheus, Grafana, Datadog, y los servicios de monitorización de los proveedores cloud (CloudWatch, etc.).

### 3.2. Métricas de Rendimiento del Modelo <a id='3.2-model-metrics'></a>

Se centran en la calidad de las predicciones del modelo.

- **Distribución de las Predicciones**: ¿Ha cambiado la distribución de las predicciones del modelo? Por ejemplo, si un modelo de fraude de repente empieza a clasificar el 90% de las transacciones como fraudulentas, algo va mal.
- **Distribución de las Características de Entrada (Data Drift)**: Comparar la distribución de cada característica en producción con su distribución en el set de entrenamiento.
- **Métricas de Precisión (Accuracy, F1, RMSE, etc.)**: **El indicador más importante**. Para calcularlas, necesitamos el **valor real (ground truth)**. Esto a menudo no está disponible de inmediato.
    - **Ejemplo**: Para predecir el churn de clientes, sabremos si un cliente realmente se ha dado de baja al final del mes. En ese momento, podemos comparar nuestra predicción con la realidad y calcular la precisión.

---

## 4. Estrategias para la Detección de Drift <a id='4-deteccion'></a>

Veamos un ejemplo práctico de cómo detectar **Data Drift**.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
import matplotlib.pyplot as plt
import seaborn as sns

# Supongamos que 'training_data' es un DataFrame con los datos de entrenamiento
np.random.seed(42)
training_data = pd.DataFrame({'income': np.random.normal(50000, 15000, 1000)})

# Y 'production_data' son los datos que llegan a nuestra API en un día
# Simularemos un Data Drift: los ingresos han aumentado
production_data = pd.DataFrame({'income': np.random.normal(65000, 18000, 200)})

# --- 4.1 Monitorización Visual de la Distribución ---
sns.kdeplot(training_data['income'], label='Training Data', fill=True)
sns.kdeplot(production_data['income'], label='Production Data', fill=True)
plt.title('Comparación de la Distribución de Ingresos')
plt.legend()
plt.show()

# --- 4.2 Uso de Tests Estadísticos (Kolmogorov-Smirnov) ---
# El test K-S compara si dos muestras provienen de la misma distribución.
# H0: Las dos muestras provienen de la misma distribución.
ks_statistic, p_value = ks_2samp(training_data['income'], production_data['income'])

print(f"Estadístico K-S: {ks_statistic:.4f}")
print(f"P-valor: {p_value:.4f}")

alpha = 0.05
if p_value < alpha:
    print("Se rechaza la hipótesis nula. Se ha detectado Data Drift.")
else:
    print("No se puede rechazar la hipótesis nula. Las distribuciones son similares.")

Este tipo de test se puede automatizar para que se ejecute periódicamente (ej. cada día) para cada característica importante, y lance una alerta si el p-valor cae por debajo de un umbral.

---

## 5. Estrategias de Re-entrenamiento <a id='5-retraining'></a>

Cuando detectamos que un modelo se ha degradado, la solución es **re-entrenarlo**.

### 5.1. ¿Cuándo Re-entrenar? <a id='5.1-cuando'></a>

- **Basado en Calendario**: La estrategia más simple. Re-entrenar el modelo cada cierto tiempo (ej. cada semana, cada mes). Es fácil de implementar pero puede ser ineficiente (re-entrenar sin necesidad) o demasiado lento (esperar un mes cuando el modelo ya se ha degradado).
- **Basado en Rendimiento**: Re-entrenar solo cuando una métrica clave (ej. accuracy, o el p-valor del test K-S) cruza un umbral predefinido. Es la estrategia más eficiente y recomendada.
- **Continuo**: Re-entrenar cada vez que llegan nuevos datos etiquetados. Puede ser costoso y complejo, reservado para casos de uso muy dinámicos.

### 5.2. Enfoques de Re-entrenamiento <a id='5.2-enfoques'></a>

- **Entrenamiento desde Cero**: Descartar el modelo antiguo y entrenar uno nuevo con datos frescos (ej. los datos del último año).
- **Entrenamiento Incremental (Fine-tuning)**: Algunos modelos (como las redes neuronales) permiten continuar el entrenamiento a partir del modelo ya entrenado, usando solo los datos nuevos. Es más rápido pero no todos los algoritmos lo soportan.

### 5.3. Diseño de un Pipeline de Re-entrenamiento <a id='5.3-pipeline'></a>

Un pipeline de re-entrenamiento automatizado es un pilar de MLOps. Usando un orquestador como **Apache Airflow** o **Kubeflow Pipelines**, los pasos serían:

1.  **Trigger**: El pipeline se inicia (por calendario o por una alerta de monitorización).
2.  **Extracción de Datos**: Recolectar los datos más recientes desde la base de datos de producción.
3.  **Validación de Datos**: Comprobar la calidad de los nuevos datos.
4.  **Entrenamiento del Modelo**: Entrenar un nuevo modelo candidato.
5.  **Evaluación y Validación**: Comparar el rendimiento del nuevo modelo con el modelo actualmente en producción usando un set de test. Si el nuevo modelo es significativamente mejor, se aprueba.
6.  **Despliegue**: El nuevo modelo se guarda en el registro de modelos y se despliega automáticamente, reemplazando al antiguo.

---

## 6. Herramientas de Monitorización <a id='6-herramientas'></a>

Existen herramientas de código abierto y comerciales especializadas en la monitorización de modelos de ML:

- **Open Source**: 
    - `Evidently AI`: Una librería de Python excelente para generar informes interactivos sobre data drift y rendimiento del modelo.
    - `MLflow`: Además de ser un registro de modelos, tiene componentes para monitorización.
    - `Prometheus` + `Grafana`: La combinación estándar para monitorización de sistemas, que se puede adaptar para métricas de modelos.

- **Comerciales**:
    - `Fiddler.ai`, `Arize AI`, `WhyLabs`.
    - Los propios servicios de los proveedores cloud: `Amazon SageMaker Model Monitor`, `Google Vertex AI Model Monitoring`.

---

## 7. Resumen y Cierre del Ciclo de MLOps <a id='7-resumen'></a>

La monitorización y el mantenimiento cierran el ciclo de vida del Machine Learning, convirtiéndolo en un proceso iterativo y robusto.

✅ Hemos aprendido que los modelos se degradan debido a cambios en el mundo real (Concept Drift y Data Drift).

✅ Sabemos qué métricas de sistema y de modelo debemos vigilar.

✅ Podemos usar técnicas visuales y estadísticas para detectar el Data Drift.

✅ Entendemos las diferentes estrategias para decidir cuándo y cómo re-entrenar nuestros modelos.

Con esto, has completado el viaje desde la idea inicial hasta un sistema de Machine Learning maduro y sostenible en producción.
